# **Assignment #2**

**Submitted By:** Piyush L. Shrestha

**Date:** February 10, 2026

## **TASK 1: Log Parser with Regex & File I/O**

Write a script that reads multiple log files in a directory, extracts all log entries with severity
ERROR or WARNING, and outputs:

- A CSV file with columns: timestamp, log level, message
- A JSON summary of counts per severity level

Handle malformed lines gracefully using regex.

In [44]:
import csv, re, json

# READING THE FILE.
file_ref = open('../data/logs/app2.log', 'r')

# PATH TO SAVE THE OUTPUT.
file_path = "../output/logs.csv"

# PATTERN TO MATCH THE FORMAT: YYYY-MM-DD HH:MM:SS [LEVEL] MESSAGE_STRING
pattern = re.compile(r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \[(INFO|WARNING|ERROR)\] (.+)$')

# COUNTER
counter = {
    'INFO': 0,
    'WARNING': 0,
    'ERROR': 0,
    'UNKNOWN': 0
} 

# WRITING ONTO FILE (ONE ROW AT A TIME).
with open(file_path, 'w', newline='') as csvfile:
    
    # HEADER OF THE CSV FILE.
    fields = ['timestamp', 'log level', 'message']
    writer = csv.DictWriter(csvfile, fieldnames=fields)
    writer.writeheader()

    for line in file_ref:
        line = line.strip()
        match = pattern.match(line)
        if match:
            ts, log, msg = match.groups()
            writer.writerow({
                fields[0]: ts,
                fields[1]: log,
                fields[2]: msg
            })
            counter[log] += 1
        else:
            counter['UNKNOWN'] += 1

# INITALLY A DICT.
print(counter, type(counter))

# CONVERTING INTO JSON. READS AS STR.
counter = json.dumps(counter)
print(counter, type(counter))

{'INFO': 16, 'WARNING': 10, 'ERROR': 8, 'UNKNOWN': 6} <class 'dict'>
{"INFO": 16, "WARNING": 10, "ERROR": 8, "UNKNOWN": 6} <class 'str'>


## **TASK 2: Structured Data Extraction & Transformation**

Given a text file containing mixed structured records (name, email, phone number), use
regex to parse valid entries and output:
- A deduplicated set of all emails
- A list of names with valid phone numbers only

Save results to a JSON file with schema { "emails": [...], "contacts": [...] }.

In [ ]:
import re, json

# READING FILE.
file_ref = open("../data/contacts.txt", "r")

# PATTERN TO CHECK THE FORMAT, AS WELL AS VALID EMAIL AND PHONE.
# VALID PHONE FORMATS: (XXX) XXX-XXX, XXX-XXX-XXX, XXX-XXXX
pattern = re.compile(
    r'^([A-Za-z]+),\s*([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}),\s*((?:\d{3}-\d{3}-\d{4}|\(\d{3}\)\s*\d{3}-\d{4}))$'
)

# TO STORE NAMES WITH VALID PHONE NUMBERS.
names = {}
# USING SET TO AVOID DUPLICATE EMAILS.
emails = set()
# USING NORMAL LIST AS CRITERIA FOR DUPLICATE PHONE NUMBERS IS NOT MENTIONED.
contacts = []

# READING & STORING VALID LINES.
for line in file_ref:
    state = pattern.match(line.strip())
    if state:
        name, email, phone = state.groups()
        names.setdefault(name, phone)
        emails.add(email)
        contacts.append(phone)

# NAMES WITH VALID CONTACTS
print("NAMES WITH VALID PHONE NUMBERS:\n")
for name, contact in names.items():
    print(name, ': ', contact)

# CREATING & WRITING INTO A JSON FILE.
with open("../output/data.json", "w") as file_w:
    
    # CREATING DATA VARIABLE TO MATCH THE SCHEMA.
    data = {
        # AS SETS ARE NOT JSON SERIALIZABLE, CONVERTING THEM INTO LISTS.
        'emails': list(emails),
        'contacts': contacts
    }
    
    # WRITING DATA INTO JSON FILE.
    file_w.write(json.dumps(data, indent=4))
    # json.dump(data, file_w, indent=2)


NAMES WITH VALID PHONE NUMBERS:

Alice :  123-456-7890
Charlie :  (555) 123-4567
Daisy :  111-222-3333


## **TASK 3: CSV Normalizer**

Write a script that reads a CSV where some fields are missing or malformed (e.g., number
fields as text). The script should:

- Detect and correct common data problems
- Fill missing numeric values with the column mean
- Save both clean CSV and a JSON report of data quality issues (counts of errors per column).

**Note:** Installing *text-to-number* package.

```python
pip install text-to-number
````

In [17]:
from text_to_number import text_to_number
print(text_to_number("Ninety "), type(text_to_number("One")))
print(text_to_number("Ninety"))
print(text_to_number("Ninety Nine"))
print(text_to_number("Ninehjhajh"))

90  <class 'str'>
90
99
Ninehjhajh


In [ ]:
import csv, json
from text_to_number import text_to_number

error_count = {
    'id': 0,
    'name': 0,
    'marks': 0
}

error_index = {
    'id': [],
    'name': [],
    'marks': []
}

# READING FILE.
with open("../data/students_raw.csv", "r") as file_r:
    
    error_log = {}
    marks = []
    valid = 0
    total = 0
    row = 1
    rows = []
    
    # READING LINES OF CSV.
    for line in csv.DictReader(file_r):
        
        # CHECKING ID, IF EMPTY SETTING THE ROW COUNT VALUE AS ID.
        try:
            id = int(line['id'].strip())
        except:
            id = row
            error_count['id'] += 1
            error_index['id'].append(row)
    
        # CHECKING NAME (IF EMPTY SETTING IT AS UNKOWN).
        name = line.get('name', '').strip().title()
        if not name:
            name = "Unknown"
            error_count['name'] += 1
            error_index['name'].append(row)

        # CHECKING MARKS, SETTING None IF NOT READABLE.
        try:
            mark = int(text_to_number(line.get("marks", ""))) 
            total += mark
            valid += 1
        except:
            mark = None
            error_count['marks'] += 1
            error_index['marks'].append(row)
            
        # UPDATING ROW COUNT.
        row += 1
        # UPDATING DATA VARIABLE.
        rows.append({
            'id': id,
            'name': name,
            'marks': mark
        })

    # MEAN OF VALID MARKS.
    avg = total//valid
    
    # UPDATING None TO MEAN VALUE.
    for r in rows:
        if r['marks'] is None:
            r['marks'] = avg

    # SAVING CSV.
    with open("../output/students.csv", "w") as file_w:
        fields = ["id", "name", "marks"]
        file_w = csv.DictWriter(file_w, fieldnames=fields)
        file_w.writeheader()
        file_w.writerows(rows)
        
    # SAVING JSON.
    with open("../output/error_log.json", "w") as file_w:
        data = {
            'error_index': error_index,
            'error_count': error_count
        }
        json.dump(data, file_w, indent=4)


## **TASK 4: Nested JSON Processor**

Given a deeply nested JSON file (user profiles with activity logs), write functions to:

- Extract all user IDs who have performed a specific action
- Count how many times each action occurred
- Output a sorted list of actions by frequency

You should use dictionary logic and comprehension.

In [59]:
import json

actions = {}
actionsPerUser = {}

def countActions(data):
    for activity in data:
        # COUNTS INDIVIDUAL ACTIONS
        actions[activity['action']] = actions.get(activity['action'], 0) + activity['count']
        # COUNTS USER BASED ACTIONS
        actionsPerUser[user['user_id']][activity['action']] = actionsPerUser[user['user_id']].get(activity['action'], 0) + activity['count']

def getSorted(data, level=1, desc = True):
    if level == 1:
        return dict(sorted(data.items(), key=lambda item: item[1], reverse = desc))
    else:
        # SORTS INNER VALUES
        return {id: dict(sorted(action.items(), key = lambda i: i[1], reverse=desc)) for id, action in sorted(data.items())}

def displayJSON(title, data, gap = 4):
    print("\n"+title+"\n")
    print(json.dumps((data), indent=gap))
    
with open('../data/users.json', 'r') as file_r:
    data = json.load(file_r)
    for user in data:
        actionsPerUser.setdefault(user['user_id'], {})
        countActions(user['profile']['activity'])
    
    displayJSON("ACTIONS PER USER", getSorted(actionsPerUser, 2, True))
    displayJSON("ACTION COUNT", getSorted(actions, True))


ACTIONS PER USER

{
    "1": {
        "logout": 83,
        "purchase": 62,
        "login": 55
    },
    "2": {
        "login": 102,
        "logout": 83,
        "purchase": 26
    },
    "3": {
        "login": 147,
        "logout": 74,
        "purchase": 59
    },
    "4": {
        "purchase": 107,
        "login": 63,
        "logout": 45
    },
    "5": {
        "login": 98,
        "purchase": 81,
        "logout": 30
    },
    "6": {
        "purchase": 113,
        "logout": 85,
        "login": 35
    },
    "7": {
        "login": 114,
        "logout": 54,
        "purchase": 31
    },
    "8": {
        "login": 112,
        "purchase": 95,
        "logout": 26
    },
    "9": {
        "purchase": 103,
        "login": 74,
        "logout": 50
    },
    "10": {
        "login": 102,
        "logout": 83,
        "purchase": 71
    },
    "11": {
        "login": 79,
        "logout": 66,
        "purchase": 50
    },
    "12": {
        "logout": 80,
        "pu

## **TASK 5. Regex Rule Validator**

Create a module with regex-based validation functions for:

- Username rules (start with a letter, no spaces, no trailing underscore)
- Password rules (min 10 chars, uppercase, digit, special char)
- Email validation
- Then write a CLI interface that reads user input CSV and validates each row’s credentials, outputting results to a new CSV.

In [3]:
import sys, csv
sys.path.append('sub_systems')
import user_validator as validator

with open('../data/credentials.csv', 'r') as file_r:
    data = csv.DictReader(file_r)
    for row in data:
        val = row['username']+', '+row['password']+', '+row['email']
        if validator.validate_user(val):
            print(f'{row['username']} has valid data.')
        else:
            print(f'{row['username']} has invalid data.')
    
    validator.view_logs()

alice1 has valid data.
1bob has invalid data.
charlie_ has invalid data.
Daisy has invalid data.

2026/13/02/15/26 09:13:51 validate_user('alice1, Password@123, alice@gmail.com') SUCCESS 'Valid CSV Format' 
2026/13/02/15/26 09:13:51 validate_username('alice1') SUCCESS 'Valid Username' 
2026/13/02/15/26 09:13:51 validate_password('Password@123') SUCCESS 'Valid Password' 
2026/13/02/15/26 09:13:51 validate_email('alice@gmail.com') SUCCESS 'Valid Email' 
2026/13/02/15/26 09:13:51 validate_user('alice1, Password@123, alice@gmail.com') SUCCESS 'Valid Inputs' 
2026/13/02/15/26 09:13:51 validate_user('1bob, short, bob@mail') SUCCESS 'Valid CSV Format' 
2026/13/02/15/26 09:13:51 validate_username('1bob') ERROR 'Invalid Username' 
2026/13/02/15/26 09:13:51 validate_user('1bob, short, bob@mail') ERROR 'Invalid Inputs' 
2026/13/02/15/26 09:13:51 validate_user('charlie_, ValidPass#99, charlie@yahoo.com') SUCCESS 'Valid CSV Format' 
2026/13/02/15/26 09:13:51 validate_username('charlie_') ERROR 'Inv

## **TASK 6: Inventory Change Tracker**

You have two CSV files representing inventory snapshots at times T1 and T2.
Write a script that:

- Reads both CSVs into dictionaries
- Outputs added, removed, and quantity-changed items

Results should be saved to JSON and printed in a human-friendly report.

In [49]:
# ASSUMING T1 IS THE PREVIOUS STOCK, AND T2 IS THE NEWER STOCK REPORT.

import csv, json

out = {}

with open('../data/inventory_t1.csv', 'r') as f1_r, open('../data/inventory_t2.csv', 'r') as f2_r:
    data_1 = csv.DictReader(f1_r)
    data_2 = csv.DictReader(f2_r)
    
    for d1, d2 in zip(data_1, data_2):
        if d1['item'] == d2['item']:
            diff = int(d1['quantity']) - int(d2['quantity'])
            state = 'Added' if d2['quantity'] > d1['quantity'] else ('Removed' if d2['quantity'] < d1['quantity'] else 'Unchanged' )
            quantity = abs(diff) if abs(diff) > 0 else int(d2['quantity'])
            print(f'{d1['item'].title()} changed to {d2['quantity']} quantities ({state.lower()} {quantity} items).')
            out.setdefault(d1['item'], {'quantity': int(d2['quantity']), 'state': 'Updated - '+state+' '+str(quantity)+' items.'})
        else:
            print(f'{d1['item'].title()} has been removed.')
            out.setdefault(d1['item'], {'quantity': 0, 'state': 'Removed '+d1['quantity']+' items.'})
            print(f'{d2['item'].title()} has been added (added {d2['quantity']} items).')
            out.setdefault(d2['item'], {'quantity': int(d2['quantity']), 'state': 'Added '+d1['quantity']+' items.'})

    print('\n'+json.dumps(out, indent=4))

Apple changed to 45 quantities (removed 5 items).
Banana changed to 30 quantities (unchanged 30 items).
Orange has been removed.
Grape has been added (added 15 items).

{
    "apple": {
        "quantity": 45,
        "state": "Updated - Removed 5 items."
    },
    "banana": {
        "quantity": 30,
        "state": "Updated - Unchanged 30 items."
    },
    "orange": {
        "quantity": 0,
        "state": "Removed 20 items."
    },
    "grape": {
        "quantity": 15,
        "state": "Added 20 items."
    }
}


## **TASK 7: Directory Report Generator**

Scan all files in a folder recursively and produce:
- A JSON file listing each file path, size, and last modified timestamp
- A set of unique file extensions
- A count of how many files per extension type

Handle permission exceptions and skip unreadable files cleanly.

In [105]:
import os, json 
from datetime import datetime
from pathlib import Path

FILE_PATH = '../data'
OUTPUT = '../output/scanned_dir_log.json'

# USING LIST FOR LOG (LATER JSON FILE) TO AVOID SAME KEY CONFLICTS.
log = []
extensions = set()
extensions_count = {}

def scan_dir(path=''):
    
    try:
        path_obj = Path(path)
        
        # FOR DIRECTORY
        if path_obj.is_dir():
            extensions.add('directory')
            extensions_count['directory'] = extensions_count.get('directory', 0) + 1
            
            try:
                stats = path_obj.stat()
                # FOR MAC/LINUX SYSTEMS
                size = str(stats.st_size) + ' bytes'
                created_date = str(datetime.fromtimestamp(stats.st_birthtime))
                modified_date = str(datetime.fromtimestamp(stats.st_mtime))
                accessible = True
            except:
                # WHEN NOT ACCESSIBLE
                created_date = None
                modified_date = None
                size = None
                accessible = False
                
            log.append({
                'name': path_obj.name,
                'type': 'directory',
                'path': str(path_obj.resolve()),
                'size': size,
                'created_at': created_date,
                'modified_at': modified_date,
                'accessible': accessible
            })
            
            # SCANNING CONTENTS OF THE DIRECTORY RECURSIVELY
            try:
                for item in path_obj.iterdir():
                    scan_dir(item)
            except Exception:
                # SIMPLY IGNORING PERMISSION ERROR OR ANY OTHER ERRORS/EXCEPTIONS.
                print(Exception)
                pass
        
        # FOR FILES
        else:
            extension = path_obj.suffix.lower()
            extensions.add(extension)
            extensions_count[extension] = extensions_count.get(extension, 0) + 1
            
            try:
                stats = path_obj.stat()
                size = str(stats.st_size) + ' bytes'
                created_date = str(datetime.fromtimestamp(stats.st_birthtime))
                modified_date = str(datetime.fromtimestamp(stats.st_mtime))
                accessible = True if os.access(path_obj, os.R_OK) else False
                
            except Exception:
                size = None
                created_date = None
                modified_date = None
                accessible = False
                            
            log.append({
                'name': path_obj.name,
                'type': 'file',
                'extension': extension,
                'path': str(path_obj.resolve()),
                'size': size,
                'created_at': created_date,
                'modified_at': modified_date,
                'accessible': accessible
            })
            
    except Exception:
        # SIMPLY IGNORING ANY ERRORS/EXCEPTIONS.
        print(Exception)
        pass
    
if Path(FILE_PATH).exists():
    scan_dir(FILE_PATH)
    
    with open(OUTPUT, 'w') as file_w:
        out = {
            'files': log,
            'extensions': list(extensions),
            'extensions_count': extensions_count
        }
        json.dump(out, file_w, indent=4)
        print(json.dumps(out, indent=4))
        print(f'Output saved to {OUTPUT}.')
else:
    print("FILE PATH DOES NOT EXIST.")


{
    "files": [
        {
            "name": "data",
            "type": "directory",
            "path": "/Users/piyushiso/Documents/Code/Sessions/AI with Python/ai-101/data",
            "size": "416 bytes",
            "created_at": "2026-02-09 12:29:48",
            "modified_at": "2026-02-09 12:29:48",
            "accessible": true
        },
        {
            "name": "students_raw.csv",
            "type": "file",
            "extension": ".csv",
            "path": "/Users/piyushiso/Documents/Code/Sessions/AI with Python/ai-101/data/students_raw.csv",
            "size": "74 bytes",
            "created_at": "2026-02-09 12:35:17",
            "modified_at": "2026-02-09 12:35:17",
            "accessible": true
        },
        {
            "name": "inventory_t2.csv",
            "type": "file",
            "extension": ".csv",
            "path": "/Users/piyushiso/Documents/Code/Sessions/AI with Python/ai-101/data/inventory_t2.csv",
            "size": "46 bytes",
    

## **TASK 8: List/Dict/Set Challenge - Analytics**

Write a program that analyzes text data from multiple files to produce:
- A list of the top 100 most common words
- A dictionary mapping each word to its count
- A set of words longer than 7 characters

Use efficient data structures and avoid loading entire files into memory at once.


**Note:** Installing *essential_generators* package to generate random paragraphs and save it into different files as the contents that we currently have is comparatively insufficient.

```python
pip install essential_generators
````

In [139]:
# CREATING TEXT FILES

from essential_generators import DocumentGenerator
from pathlib import Path

PATH = '../data/generated'
OUTPUTS = ['gen1.txt', 'gen2.txt', 'gen3.txt']

gen = DocumentGenerator()

for output in OUTPUTS:
    Path(PATH).mkdir(parents=True, exist_ok=True)
    with open(PATH+'/'+output, 'w') as file_w:
        # ADD 20 RANDOM PARAGRAPHS TO THE FILE.
        file_w.writelines(gen.paragraph() + '\n' for _ in range(20))
        print("Wrote in "+PATH+'/'+output)

Wrote in ../data/generated/gen1.txt
Wrote in ../data/generated/gen2.txt
Wrote in ../data/generated/gen3.txt


In [140]:
import os, re
# Utilizing Counter to make couting and finding top words easily.
from collections import Counter

DIR = '../data/generated'

words_counter = Counter()
long_words = set()

pattern = re.compile(r'\b\w+\b')

for file in os.listdir(DIR):
    path = os.path.join(DIR, file)
    if os.path.isfile(path):
        with open(path, 'r') as file_r:
            for line in file_r:
                words = pattern.findall(line.lower())
                for word in words:
                    words_counter[word] += 1
                    if len(word) > 7:
                        long_words.add(word)

print("\nTop 100 Most Common Words:\n\n", words_counter.most_common(100))
print("\nWord Counts:\n\n", words_counter)
print("\nWords Having More Than 7 Characters:\n\n", long_words)
                


Top 100 Most Common Words:

 [('the', 238), ('of', 129), ('and', 119), ('in', 98), ('to', 85), ('a', 75), ('that', 36), ('as', 35), ('are', 34), ('by', 34), ('is', 34), ('for', 31), ('s', 30), ('or', 29), ('with', 26), ('was', 23), ('an', 21), ('which', 20), ('from', 20), ('on', 19), ('has', 16), ('such', 15), ('at', 14), ('have', 14), ('this', 14), ('other', 13), ('it', 11), ('some', 11), ('new', 11), ('one', 11), ('its', 11), ('world', 10), ('more', 10), ('also', 10), ('be', 9), ('over', 9), ('many', 9), ('largest', 9), ('south', 9), ('including', 9), ('history', 9), ('they', 9), ('were', 8), ('public', 8), ('10', 8), ('through', 8), ('had', 8), ('being', 8), ('not', 8), ('8', 7), ('2', 7), ('5', 7), ('social', 7), ('however', 7), ('i', 7), ('where', 7), ('sources', 7), ('japan', 7), ('can', 7), ('their', 7), ('1', 7), ('state', 7), ('include', 7), ('government', 6), ('earth', 6), ('into', 6), ('information', 6), ('human', 6), ('germany', 6), ('first', 6), ('e', 6), ('de', 6), ('bee

## **TASK 9: JSON-to-CSV Converter with Mapping**

Build a script that takes multiple JSON files (each with potentially different keys) and:

- Extracts a unified set of fields based on a mapping config
- Outputs a combined CSV with consistent columns

Your script should dynamically handle missing keys per record.

## **Task 10: Regex-Driven Data Cleaner**

Write a program that:

- Reads a large unstructured text file
- Extracts only valid IP addresses, dates, and email addresses using regex
- Aggregates these into separate CSV files
- Outputs a final JSON report of how many of each type were found

Handle overlapping matches and avoid partial capture conflicts.

In [164]:
import os, re, copy
from pathlib import Path
from collections import Counter

FILE_PATH = '../data/raw_dump.txt'
OUTPUT_PATH = '../output'
OUTPUTS = ['ips.csv', 'dates.csv', 'emails.csv']

count = Counter()

# REGEX
ip_pattern = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
date_pattern = re.compile(r'\b\d{4}-\d{2}-\d{2}\b|\b\d{2}/\d{2}/\d{4}\b')
email_pattern = re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')

# LISTS TO HOLD VALID DATA
ips = []
dates = []
emails = []

with open(FILE_PATH, 'r') as file_r:
    for line in file_r:
        ips.extend(ip_pattern.findall(line))
        dates.extend(date_pattern.findall(line))
        emails.extend(email_pattern.findall(line))
        
    count['ips'] += len(ips)
    count['dates'] += len(dates)
    count['emails'] += len(emails)
    print(ips, dates, emails, count, sep='\n')
    
for file in OUTPUTS:
    if os.path.exists(OUTPUT_PATH):
        file_path = os.path.join(OUTPUT_PATH, file)
        field = ''
        values = []
        with open(file_path, 'w') as file_w:
            if file == 'ips.csv':
                field = 'ip'
                values = copy.deepcopy(ips)
            elif file == 'dates.csv':
                field = 'date'
                values = copy.deepcopy(dates)
            else:
                field = 'email'
                values = copy.deepcopy(emails)
            try:
                csv_w = csv.DictWriter(file_w, fieldnames=[field])
                csv_w.writeheader()
                rows = [{field: v} for v in values]
                csv_w.writerows(rows)
                print(f"UPDATED FILE ({file_path}).")
            except:
                print(f"ERROR UPDATING FILE ({file_path}).")
    else:
        print(f"FILE DIRECTORY DOES NOT EXIST ({OUTPUT_PATH}).")

['192.168.1.1', '10.0.0.25']
['2024-10-05', '05/12/2023']
['admin@example.com', 'support@mail.org']
Counter({'ips': 2, 'dates': 2, 'emails': 2})
UPDATED FILE (../output/ips.csv).
UPDATED FILE (../output/dates.csv).
UPDATED FILE (../output/emails.csv).
